# 🏦 Analyse du Risque Crédit Bancaire

**Auteure :** Lavinia Bulgarean-Haiduc  
**Stack :** Python · Pandas · NumPy  
**Dataset :** German Credit Data (source : Kaggle)  
**Objectif :** Explorer, nettoyer et préparer un dataset de 1000 clients bancaires en vue d'évaluer un modèle de prédiction de la durée des prêts.

---

## 📋 Contexte

Ce projet porte sur un jeu de données de **1000 clients** ayant souscrit à un prêt bancaire.  
Chaque client est décrit par plusieurs caractéristiques : âge, type de logement, montant du crédit, durée du prêt, type de compte épargne, etc.

Le projet couvre l'ensemble du pipeline d'un Data Analyst :
- Exploration et description des données
- Nettoyage et gestion des valeurs manquantes
- Encodage des variables catégorielles
- Catégorisation de variables numériques
- Fusion de datasets
- Évaluation d'un modèle de régression

## 1. 📦 Import des librairies et chargement des données

In [ ]:
import pandas as pd

# Chargement du dataset
df = pd.read_csv('german_credit_data.csv', sep=';', index_col=0)

# Aperçu des premières lignes
df.head()

## 2. 🔍 Exploration initiale

In [ ]:
# Structure et types des variables
df.info()

In [ ]:
# Montant maximal du crédit par motif de prêt
max_credit = df.groupby('purpose')['credit_amount'].max()
print('Montant maximal du crédit par motif :')
print(max_credit)

In [ ]:
# Motif de prêt le plus fréquent
motif_frequent = df['purpose'].value_counts().idxmax()
print(f'Motif le plus fréquent : {motif_frequent}')

In [ ]:
# Profil de l'emprunteur le plus âgé
print('--- Emprunteur le plus âgé ---')
plus_age = df.loc[df['age'].idxmax()]
print(plus_age)

# Profil de l'emprunteur le plus jeune
print('\n--- Emprunteur le plus jeune ---')
plus_jeune = df.loc[df['age'].idxmin()]
print(plus_jeune)

## 3. 🛠️ Préparation et nettoyage des données

In [ ]:
# Conversion de la variable 'duration' : suppression de '-month' et conversion en int
# Ex : '24-month' -> 24
df['duration'] = df['duration'].apply(lambda x: int(x[:-6]))

print('Variable duration convertie en numérique :')
df['duration'].head()

In [ ]:
# Création d'une variable catégorielle 'duration_categ'
# court-terme  : durée <= 10 mois
# moyen-terme  : 10 < durée <= 30 mois
# long-terme   : durée > 30 mois

df['duration_categ'] = 'court-terme'
df.loc[(df['duration'] > 10) & (df['duration'] <= 30), 'duration_categ'] = 'moyen-terme'
df.loc[df['duration'] > 30, 'duration_categ'] = 'long-terme'

print('Répartition des catégories de durée :')
print(df['duration_categ'].value_counts())

In [ ]:
# Catégorie de prêt la plus courante chez les locataires
locataires = df[df['housing'] == 'rent']
cat_locataires = locataires['duration_categ'].value_counts().idxmax()
print(f'Catégorie la plus courante chez les locataires : {cat_locataires}')

## 4. 🔢 Encodage des variables catégorielles

In [ ]:
# Encodage ordinal de 'saving_accounts' et 'checking_account'
# little=0, moderate=1, quite rich=2, rich=3

encoding_map = {
    'little': 0,
    'moderate': 1,
    'quite rich': 2,
    'rich': 3
}

df['saving_accounts'].replace(encoding_map, inplace=True)
df['checking_account'].replace(encoding_map, inplace=True)

print('Encodage effectué. Aperçu :')
df[['saving_accounts', 'checking_account']].head()

## 5. 🩹 Gestion des valeurs manquantes

In [ ]:
# Comptage des valeurs manquantes par colonne
print('Valeurs manquantes par colonne :')
print(df.isna().sum())

In [ ]:
# Remplacement des valeurs manquantes de 'housing' par le mode (valeur la plus fréquente)
# Justification : variable catégorielle -> utilisation du mode
df['housing'] = df['housing'].fillna(df['housing'].mode()[0])

# Remplacement des valeurs manquantes de 'age' par la moyenne
# Justification : variable numérique continue -> utilisation de la moyenne
df['age'] = df['age'].fillna(df['age'].mean())

print('Valeurs manquantes après traitement :')
print(df.isna().sum())

## 6. 🔗 Fusion avec les prédictions du modèle

In [ ]:
# Chargement des prédictions d'un modèle de régression pré-entraîné
pred_german = pd.read_csv('predictions_german.csv', sep=';', index_col=0)

# Fusion avec le dataset principal sur l'index client
df_pred = df.merge(pred_german, left_index=True, right_index=True)

print(f'Dataset fusionné : {df_pred.shape[0]} lignes, {df_pred.shape[1]} colonnes')
df_pred.head()

## 7. 📊 Évaluation du modèle de régression

In [ ]:
# Calcul de l'erreur entre durée réelle et durée prédite
df_pred['error'] = df_pred['duration'] - df_pred['predictions']

print('Aperçu des erreurs de prédiction :')
df_pred[['duration', 'predictions', 'error']].head(10)

In [ ]:
# Nombre de clients dont la durée de prêt a été surestimée par le modèle
nb_surestimes = (df_pred['predictions'] > df_pred['duration']).sum()
pct_surestimes = nb_surestimes / len(df_pred) * 100

print(f'Clients avec durée surestimée : {nb_surestimes} ({pct_surestimes:.1f}%)')
print(f'Clients avec durée sous-estimée ou exacte : {len(df_pred) - nb_surestimes} ({100 - pct_surestimes:.1f}%)')

## 8. 📝 Conclusions

Ce projet a permis de réaliser un pipeline complet d'analyse de données :

- **Exploration** : le motif le plus fréquent est l'achat de voiture, et les vacances représentent les montants les plus élevés.
- **Nettoyage** : 76 valeurs manquantes sur l'âge (remplacées par la moyenne) et 45 sur le logement (remplacées par le mode).
- **Catégorisation** : la majorité des prêts sont à moyen terme (10-30 mois), notamment chez les locataires.
- **Évaluation du modèle** : le modèle surestime la durée pour environ 53% des clients, ce qui indique un biais à la hausse à corriger dans une prochaine itération.

---
*Projet réalisé dans le cadre de ma formation Data Analyst · [madamedatait.com](https://madamedatait.com)*